<a href="https://colab.research.google.com/github/mbudisic/test/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audio Classification: Violin vs. Non-Violin

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install Libraries

In [ ]:
!pip install torchaudio torchvision torchaudio scikit-learn
!apt-get update && apt-get install -y ffmpeg


## Import Libraries

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import os
import glob
import random
import numpy as np

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchaudio Version: {torchaudio.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Prepare Data

In [ ]:
import os
import glob
from sklearn.model_selection import train_test_split

BASE_DIR = '/content/drive/My Drive/audio_classification_data'
v_dir = os.path.join(BASE_DIR, 'violin')
nv_dir = os.path.join(BASE_DIR, 'no_violin')

v_files = glob.glob(os.path.join(v_dir, '*.mp4'))
nv_files = glob.glob(os.path.join(nv_dir, '*.mp4'))

data = []
for f in v_files:
    data.append((f, 1))
for f in nv_files:
    data.append((f, 0))

train_val_files, test_files = train_test_split(data, test_size=0.1, random_state=42, stratify=[d[1] for d in data])
train_files, val_files = train_test_split(train_val_files, test_size=(0.1/0.9), random_state=42, stratify=[d[1] for d in train_val_files])

## PyTorch Dataset & DataLoaders

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

class AudioDataset(Dataset):
    def __init__(self, data_list, sample_rate=16000, max_len_sec=5):
        self.data_list = data_list
        self.sample_rate = sample_rate
        self.max_len_samples = int(sample_rate * max_len_sec)

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        filepath, label = self.data_list[idx]

        try:
            waveform, sr = torchaudio.load(filepath)
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            if sr != self.sample_rate:
                resampler = T.Resample(sr, self.sample_rate)
                waveform = resampler(waveform)

            if waveform.shape[1] < self.max_len_samples:
                padding = self.max_len_samples - waveform.shape[1]
                waveform = F.pad(waveform, (0, padding))
            elif waveform.shape[1] > self.max_len_samples:
                waveform = waveform[:, :self.max_len_samples]

            return waveform, torch.tensor(label, dtype=torch.long)

        except Exception as e:
            print(f"Error loading audio file {filepath}: {e}")
            return torch.zeros(1, self.max_len_samples), torch.tensor(-1, dtype=torch.long)

train_dataset = AudioDataset(train_files)
val_dataset = AudioDataset(val_files)
test_dataset = AudioDataset(test_files)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


## Neural Network Model (Wav2Vec2 Fine-tuning)

In [ ]:
import torch.nn as nn
import torchaudio.models as models
import torch

class Wav2Vec2Classifier(nn.Module):
    def __init__(self, num_classes=2, freeze_feature_extractor=False):
        super(Wav2Vec2Classifier, self).__init__()
        self.feature_extractor = models.wav2vec2_base(pretrained=True)

        if freeze_feature_extractor:
            for param in self.feature_extractor.parameters():
                param.requires_grad = False

        self.classifier = nn.Linear(self.feature_extractor.encoder.config.hidden_size, num_classes)

    def forward(self, x):
        if x.dim() == 3:
            x = x.squeeze(1)
        features, _ = self.feature_extractor(x)
        pooled_features = features.mean(dim=1)
        logits = self.classifier(pooled_features)
        return logits

model = Wav2Vec2Classifier(num_classes=2, freeze_feature_extractor=False).to(device)


## Training Setup

In [ ]:
import torch.optim as optim
from tqdm.notebook import tqdm

learning_rate = 0.001
num_epochs = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    model.train()
    best_val_accuracy = 0.0

    for epoch in range(num_epochs):
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} Training"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        train_accuracy = 100 * correct_train / total_train

        model.eval()
        correct_val = 0
        total_val = 0
        val_loss = 0.0
        with torch.no_grad():
            for inputs_val, labels_val in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} Validation"):
                inputs_val, labels_val = inputs_val.to(device), labels_val.to(device)
                outputs_val = model(inputs_val)
                loss_val = criterion(outputs_val, labels_val)
                val_loss += loss_val.item()
                _, predicted_val = torch.max(outputs_val.data, 1)
                total_val += labels_val.size(0)
                correct_val += (predicted_val == labels_val).sum().item()

        val_accuracy = 100 * correct_val / total_val

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), 'best_audio_classifier.pth')

        model.train()

if len(train_files) > 0 and len(val_files) > 0:
    train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device)


## Model Evaluation

In [ ]:
import os
import torch
from tqdm.notebook import tqdm

def evaluate_model(model, test_loader, device):
    model.eval()
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_accuracy = 100 * correct_test / total_test
    print(f"Test Accuracy: {test_accuracy:.2f}%")

if os.path.exists('best_audio_classifier.pth'):
    model.load_state_dict(torch.load('best_audio_classifier.pth'))
    evaluate_model(model, test_loader, device)


## Prepare a Simple Test Dataset (MiniSpeechCommands)

This section will download the MiniSpeechCommands dataset and organize a subset of its files into your Google Drive, mimicking the `violin` / `no_violin` structure. We'll use 'yes' as one class and 'no' as the other.

**Note:** This will download WAV files, so you'll need to update the `glob` pattern in cell `46e1c616` from `*.mp4` to `*.wav` after running these cells.

In [ ]:
import torchaudio
import os
import shutil

# Define the root directory for the dataset on Colab's local filesystem
DATASET_PATH = './speech_commands'

# Download the MiniSpeechCommands dataset
print("Downloading MiniSpeechCommands dataset...")
dataset = torchaudio.datasets.SPEECHCOMMANDS(root=DATASET_PATH, url="_V0.02", download=True)
print("Download complete.")

# Define your Google Drive base directory (ensure it matches BASE_DIR in cell 46e1c616)
GD_BASE_DIR = '/content/drive/My Drive/audio_classification_data'
GD_CLASS_1_DIR = os.path.join(GD_BASE_DIR, 'violin') # Use 'violin' as target for 'yes'
GD_CLASS_0_DIR = os.path.join(GD_BASE_DIR, 'no_violin') # Use 'no_violin' as target for 'no'

os.makedirs(GD_CLASS_1_DIR, exist_ok=True)
os.makedirs(GD_CLASS_0_DIR, exist_ok=True)

print(f"Organizing 'yes' and 'no' samples into {GD_CLASS_1_DIR} and {GD_CLASS_0_DIR}...")

# Iterate through the downloaded dataset and copy files
# We'll use 'yes' for the 'violin' class (label 1) and 'no' for the 'no_violin' class (label 0)
for i in range(len(dataset)):
    waveform, sample_rate, label_text, speaker_id, utterance_number = dataset[i]

    if label_text == 'yes':
        target_dir = GD_CLASS_1_DIR
    elif label_text == 'no':
        target_dir = GD_CLASS_0_DIR
    else:
        continue # Skip other words

    # The actual path of the WAV file in the downloaded dataset
    # This part requires knowing the internal structure of SPEECHCOMMANDS dataset
    # For V0.02, it's usually `root/SpeechCommands/speech_commands_v0.02/{label_text}/{filename}.wav`
    source_file_name = f"{label_text}_{speaker_id}_{utterance_number}.wav"
    source_file_path = os.path.join(DATASET_PATH, 'SpeechCommands', 'speech_commands_v0.02', label_text, source_file_name)

    if os.path.exists(source_file_path):
        destination_path = os.path.join(target_dir, source_file_name)
        shutil.copy(source_file_path, destination_path)

print("Dataset preparation complete in Google Drive.")

# Clean up local download to save space (optional)
# shutil.rmtree(DATASET_PATH)

### Important: Update Data Loading

Since we've prepared WAV files, you **must update** cell `46e1c616` to look for `*.wav` files instead of `*.mp4`. Modify the following lines in cell `46e1c616`:

```python
v_files = glob.glob(os.path.join(v_dir, '*.wav')) # Change from *.mp4
nv_files = glob.glob(os.path.join(nv_dir, '*.wav')) # Change from *.mp4
```

After making this change, you can re-run cell `46e1c616` and all subsequent cells to use this new dataset.

## Demonstrating MusicNet Dataset Loading (Standalone)

This section shows how to load the MusicNet dataset, as requested. It is **not** integrated into the main audio classification workflow above.

In [ ]:
import torchaudio
import os

# Define a local directory for MusicNet download
MUSICNET_DATASET_PATH = './musicnet_data'

print(f"Downloading and loading MusicNet dataset to {MUSICNET_DATASET_PATH}...")

# Initialize the MusicNet dataset
# The dataset will be downloaded if it's not already present.
musicnet_dataset = torchaudio.datasets.MUSICNET(root=MUSICNET_DATASET_PATH, download=True)

print("MusicNet dataset loaded successfully!")
print(f"Number of samples in MusicNet dataset: {len(musicnet_dataset)}")

# You can access individual samples like this:
# waveform, sample_rate, labels = musicnet_dataset[0]
# print(f"Sample 0 - Waveform shape: {waveform.shape}, Sample Rate: {sample_rate}, Labels: {labels}")

In [ ]:
print("\nLoading 'train' subset...")
musicnet_train_dataset = torchaudio.datasets.MUSICNET(root=MUSICNET_DATASET_PATH, download=False, subset='train')
print(f"Number of samples in MusicNet 'train' subset: {len(musicnet_train_dataset)}")

print("\nLoading 'test' subset...")
musicnet_test_dataset = torchaudio.datasets.MUSICNET(root=MUSICNET_DATASET_PATH, download=False, subset='test')
print(f"Number of samples in MusicNet 'test' subset: {len(musicnet_test_dataset)}")